# 📣 Notebook 1 — Event-Driven vs Request-Driven

> **Goal:** Build an intuition for *why* we'd ever prefer sending an event over making a direct API call.

## 🏪 A real-world analogy

Imagine a busy coffee shop.

- **Request-driven** is like the cashier walking to the barista, handing over an order, and **standing there waiting** until the coffee is ready. If the barista is slow, the cashier can't take the next customer. If a new "whipped-cream specialist" joins the team, the cashier has to learn they exist and walk to them too.
- **Event-driven** is like the cashier sticking an order ticket on the rail and **walking back to the register immediately**. The barista grabs tickets when ready. If you add a whipped-cream specialist tomorrow, they just start reading the same rail — the cashier doesn't change a thing.

The rail is the **event bus**. The ticket is the **event**. The cashier is the **producer**. The barista is a **consumer**.


## 🛠️ Setup

```bash
cd 05-microservices/event-driven-architecture
uv sync
```

Then in VS Code, select the `.venv` kernel (top-right of the notebook).
If it isn't listed, reload the window: `Cmd+Shift+P` → **Reload Window**.


## 1️⃣ The "bad" way: request-driven with hard-coded calls

Let's model placing an order. When an order is placed, three things must happen: ship it, email a receipt, record analytics.

In the naive version the order function **knows every downstream service by name** and calls them one by one.


In [ ]:
def ship(order):      print(f"  📦 shipping        {order}")
def email(order):     print(f"  ✉️  emailing receipt {order}")
def analytics(order): print(f"  📊 analytics        {order}")

def place_order_RPC(order):
    # ❌ The producer is glued to every consumer.
    # Adding a 4th thing (say, fraud detection) means editing THIS function.
    ship(order)
    email(order)
    analytics(order)

place_order_RPC("o-1")

### Why this hurts at scale
1. **Tight coupling.** Every new feature = edit the producer + re-deploy it.
2. **Cascading failures.** If `email` raises, `analytics` never runs.
3. **Slow.** The user waits for *everything* before getting a response.
4. **Hard to reuse.** Another team that also cares about "an order happened" has to import your code.


## 2️⃣ The "good" way: event-driven with a tiny event bus

Now the producer just **announces a fact**: *"an order was placed"*. It has no idea who's listening.

Consumers **subscribe** to the fact they care about. Adding a new consumer = writing a new function. The producer never changes.


In [ ]:
from collections import defaultdict

class EventBus:
    """The smallest useful event bus: a dict of {event_name: [handlers]}."""
    def __init__(self):
        self.subs = defaultdict(list)

    def subscribe(self, event_name, handler):
        self.subs[event_name].append(handler)

    def publish(self, event_name, payload):
        # The producer doesn't know who (if anyone) is listening.
        for handler in self.subs[event_name]:
            handler(payload)

bus = EventBus()
bus.subscribe("order_placed", ship)
bus.subscribe("order_placed", email)
bus.subscribe("order_placed", analytics)

def place_order_EVENT(order):
    # ✅ Producer only knows the *event name*. That's the entire contract.
    bus.publish("order_placed", order)

place_order_EVENT("o-2")

## 3️⃣ Proof of decoupling: add a feature without touching the producer

Marketing wants a loyalty-points feature. In the RPC version we'd edit `place_order_RPC`.
In the event-driven version we just **subscribe a new handler**.


In [ ]:
def loyalty(order): print(f"  ⭐ +10 pts for  {order}")

bus.subscribe("order_placed", loyalty)

# Same producer call as before — but now 4 things happen.
place_order_EVENT("o-3")

## 🧠 Key takeaways

| Dimension            | Request-driven            | Event-driven                     |
|----------------------|---------------------------|----------------------------------|
| Who knows whom       | Producer knows consumers  | Producer knows only event names  |
| Adding a consumer    | Edit producer             | Write a new subscriber           |
| Failure blast radius | One bad call breaks chain | Each subscriber fails in isolation *(not yet — nb 2)* |
| Latency              | Sum of all calls          | Producer returns immediately *(not yet — nb 2)* |

> ⚠️ **Be precise about what we just proved.** The bus above is *synchronous* — it
> calls each handler inline, in the producer's thread. So we have demonstrated the
> **decoupling** benefit (rows 1 and 2) and nothing else: a crashing subscriber still
> kills the producer, and the producer still waits for every handler. Those last two
> rows are properties of a *broker*, not of pub/sub as an idea, and Notebook 2 earns
> them with a queue and a worker thread.

## ⚠️ Nothing is free

Event-driven systems also introduce:
- **Implicit flow** — you can't just Ctrl+click to see "what happens next".
- **Schema is now a public contract** — changing it breaks invisible subscribers.
- **Delivery guarantees** — "at-least-once" means consumers must be *idempotent*.

We'll tackle each of these in the next notebooks.

**Next up:** [`02_worked_example.ipynb`](./02_worked_example.ipynb) — make the bus asynchronous and fault-isolated.
